In [1]:
import os
import sys
sys.path.append("../../")
from callmefair.util.fair_util import BMInterface
from callmefair.mitigation.fair_grid import BMGridSearch
from callmefair.mitigation.fair_bm import BMType
import pandas as pd
import time
T0 = time.time()  # only grid-search logs written after this point are reported at the end

# (the original `import mkl; mkl.set_num_threads(5)` is Intel-only and was removed)
N_THREADS = min(8, os.cpu_count() or 1)

df = pd.read_csv('../data/wids_pre/wids.csv')

# Defining the name of the label column
label_name = 'DiagPeriodL90D'
# Defininf the name of the privileged group
sensitive_attribute = ['patient_age']

bmI = BMInterface(df, label_name, sensitive_attribute)

privileged_group = [{'patient_age': 1}]
unprivileged_group = [{'patient_age': 0}]

pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


pip install 'aif360[Reductions]'


In [2]:
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC  
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier

mlp= MLPClassifier(
    max_iter=500,
    hidden_layer_sizes = (200,200,200,200),
    activation = 'logistic',
    solver = 'adam',
    alpha = 0.01,
    learning_rate = 'adaptive'
)

# Create a new instance of the classifier
lr = LogisticRegression(solver='liblinear')

xgbr =  XGBClassifier(
    max_depth=8,
    learning_rate=0.01,
    gamma = 0.25,
    n_estimators = 500,
    subsample = 0.8,
    colsample_bytree = 0.3)

tabnet = TabNetClassifier()

cat = CatBoostClassifier(eval_metric='Accuracy',
                         depth =  4,
                         learning_rate = 0.01,
                         iterations = 10)


model_list = [tabnet, xgbr, cat, mlp, lr]

In [3]:
tabnet.__str__()#.startswith('XGBClassifier')

"TabNetClassifier(n_d=8, n_a=8, n_steps=3, gamma=1.3, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=2, n_shared=2, epsilon=1e-15, momentum=0.02, lambda_sparse=0.001, seed=0, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=None, scheduler_params={}, mask_type='sparsemax', input_dim=None, output_dim=None, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[])"

In [4]:
from itertools import product
from more_itertools import powerset

pre_bm = [BMType.preReweighing,BMType.preDisparate]
pos_bm = [BMType.posCalibrated, BMType.posEqqOds, BMType.posROC]

bm_list = [[pre, pos] for pre, pos in product(pre_bm, pos_bm)]
bm_list.extend([p] for p in pre_bm); bm_list.extend([p] for p in pos_bm)
bm_list

[[<BMType.preReweighing: 1>, <BMType.posCalibrated: 6>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>],
 [<BMType.preDisparate: 2>, <BMType.posCalibrated: 6>],
 [<BMType.preDisparate: 2>, <BMType.posEqqOds: 7>],
 [<BMType.preDisparate: 2>, <BMType.posROC: 8>],
 [<BMType.preReweighing: 1>],
 [<BMType.preDisparate: 2>],
 [<BMType.posCalibrated: 6>],
 [<BMType.posEqqOds: 7>],
 [<BMType.posROC: 8>]]

In [5]:
import copy

bm_list_in = copy.deepcopy(bm_list)
[l.append(BMType.inAdversarial) for l in bm_list_in]
bm_list_in.extend(bm_list)
bm_list_in.append([BMType.inAdversarial])
bm_list_in

[[<BMType.preReweighing: 1>,
  <BMType.posCalibrated: 6>,
  <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>,
  <BMType.posCalibrated: 6>,
  <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.inAdversarial: 4>],
 [<BMType.preDisparate: 2>, <BMType.inAdversarial: 4>],
 [<BMType.posCalibrated: 6>, <BMType.inAdversarial: 4>],
 [<BMType.posEqqOds: 7>, <BMType.inAdversarial: 4>],
 [<BMType.posROC: 8>, <BMType.inAdversarial: 4>],
 [<BMType.preReweighing: 1>, <BMType.posCalibrated: 6>],
 [<BMType.preReweighing: 1>, <BMType.posEqqOds: 7>],
 [<BMType.preReweighing: 1>, <BMType.posROC: 8>],
 [<BMType.preDisparate: 2>, <BMType.posCalibrated: 6>],
 [<BMType.preDisparate: 

In [6]:
for model in model_list:
    bmG = BMGridSearch(bmI, model=model, bm_list=bm_list, privileged_group=privileged_group, unprivileged_group=unprivileged_group)
    bmG.run_single_sensitive()

epoch 0  | loss: 0.66987 | val_0_auc: 0.7171  |  0:00:00s


epoch 1  | loss: 0.6114  | val_0_auc: 0.75032 |  0:00:00s


epoch 2  | loss: 0.54285 | val_0_auc: 0.75395 |  0:00:01s


epoch 3  | loss: 0.50212 | val_0_auc: 0.78006 |  0:00:01s


epoch 4  | loss: 0.49356 | val_0_auc: 0.76958 |  0:00:02s


epoch 5  | loss: 0.48733 | val_0_auc: 0.77362 |  0:00:02s


... (314 outputs left out)

0:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 1.3ms	remaining: 11.7ms
1:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 2.63ms	remaining: 10.5ms
2:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 3.98ms	remaining: 9.29ms
3:	learn: 0.8131032	test: 0.8254092	best: 0.8254092 (0)	total: 5.34ms	remaining: 8.01ms
4:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 6.58ms	remaining: 6.58ms
5:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 7.87ms	remaining: 5.25ms
6:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 9.17ms	remaining: 3.93ms
7:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 10.5ms	remaining: 2.63ms
8:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 11.9ms	remaining: 1.32ms
9:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 13.2ms	remaining: 0us

bestTest = 0.8254091972
bestIteration = 0

Shrink model to first 1 iterations.


0:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 1.25ms	remaining: 11.3ms
1:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 2.59ms	remaining: 10.4ms
2:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 3.95ms	remaining: 9.21ms
3:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 5.26ms	remaining: 7.88ms
4:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 6.59ms	remaining: 6.59ms
5:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 7.91ms	remaining: 5.27ms
6:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 9.38ms	remaining: 4.02ms
7:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 10.7ms	remaining: 2.67ms
8:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 12.1ms	remaining: 1.34ms
9:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 13.4ms	remaining: 0us

bestTest = 0.8254091972
bestIteration = 0

Shrink model to first 1 iterations.


0:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 1.18ms	remaining: 10.6ms
1:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 2.55ms	remaining: 10.2ms
2:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 3.81ms	remaining: 8.89ms
3:	learn: 0.8131032	test: 0.8254092	best: 0.8254092 (0)	total: 5.24ms	remaining: 7.85ms
4:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 6.46ms	remaining: 6.46ms
5:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 7.82ms	remaining: 5.21ms
6:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 9.22ms	remaining: 3.95ms
7:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 10.6ms	remaining: 2.65ms
8:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 11.9ms	remaining: 1.32ms
9:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 13.4ms	remaining: 0us

bestTest = 0.8254091972
bestIteration = 0

Shrink model to first 1 iterations.
0:	learn: 0.8132007	test: 0.8254092	best: 0

0:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 1.13ms	remaining: 10.1ms
1:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 2.48ms	remaining: 9.93ms
2:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 3.72ms	remaining: 8.69ms
3:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 5.05ms	remaining: 7.58ms
4:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 6.42ms	remaining: 6.42ms
5:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 7.71ms	remaining: 5.14ms
6:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 9.05ms	remaining: 3.88ms
7:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 10.5ms	remaining: 2.62ms
8:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 11.8ms	remaining: 1.31ms
9:	learn: 0.8132007	test: 0.8254092	best: 0.8254092 (0)	total: 13.1ms	remaining: 0us

bestTest = 0.8254091972
bestIteration = 0

Shrink model to first 1 iterations.


In [7]:
bmG = BMGridSearch(bmI, model=None, bm_list=[[BMType.inAdversarial]], privileged_group=privileged_group, unprivileged_group=unprivileged_group)
bmG.run_single_sensitive()

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


epoch 0; iter: 0; batch classifier loss: 0.718934
epoch 1; iter: 0; batch classifier loss: 0.484462
epoch 2; iter: 0; batch classifier loss: 0.515218
epoch 3; iter: 0; batch classifier loss: 0.512658


epoch 4; iter: 0; batch classifier loss: 0.429878


epoch 5; iter: 0; batch classifier loss: 0.500581
epoch 6; iter: 0; batch classifier loss: 0.493575
epoch 7; iter: 0; batch classifier loss: 0.511725
epoch 8; iter: 0; batch classifier loss: 0.568001
epoch 9; iter: 0; batch classifier loss: 0.467371


epoch 10; iter: 0; batch classifier loss: 0.415858


... (23 outputs left out)

epoch 36; iter: 0; batch classifier loss: 1.013798; batch adversarial loss: 0.108525
epoch 37; iter: 0; batch classifier loss: 1.136420; batch adversarial loss: 0.035760
epoch 38; iter: 0; batch classifier loss: 1.079911; batch adversarial loss: 0.101675
epoch 39; iter: 0; batch classifier loss: 0.865307; batch adversarial loss: 0.080570


epoch 40; iter: 0; batch classifier loss: 0.816291; batch adversarial loss: 0.024551
epoch 41; iter: 0; batch classifier loss: 0.595748; batch adversarial loss: 0.024124
epoch 42; iter: 0; batch classifier loss: 0.578008; batch adversarial loss: 0.020833
epoch 43; iter: 0; batch classifier loss: 0.780385; batch adversarial loss: 0.095973


epoch 44; iter: 0; batch classifier loss: 0.711179; batch adversarial loss: 0.093855
epoch 45; iter: 0; batch classifier loss: 0.706544; batch adversarial loss: 0.022138
epoch 46; iter: 0; batch classifier loss: 0.788334; batch adversarial loss: 0.095075
epoch 47; iter: 0; batch classifier loss: 0.659819; batch adversarial loss: 0.091887


epoch 48; iter: 0; batch classifier loss: 0.676079; batch adversarial loss: 0.018752
epoch 49; iter: 0; batch classifier loss: 0.669046; batch adversarial loss: 0.017331


In [8]:
# The original looped `for bm in bm_list` while ignoring `bm`, re-running the identical
# grid 11 times; a single run is performed here.
bmG = BMGridSearch(bmI, model=None, bm_list=bm_list_in, privileged_group=privileged_group, unprivileged_group=unprivileged_group)
bmG.run_single_sensitive()

epoch 0; iter: 0; batch classifier loss: 0.644991
epoch 1; iter: 0; batch classifier loss: 0.501446
epoch 2; iter: 0; batch classifier loss: 0.444613
epoch 3; iter: 0; batch classifier loss: 0.554672
epoch 4; iter: 0; batch classifier loss: 0.478175


epoch 5; iter: 0; batch classifier loss: 0.518375
epoch 6; iter: 0; batch classifier loss: 0.425650
epoch 7; iter: 0; batch classifier loss: 0.441952
epoch 8; iter: 0; batch classifier loss: 0.525099
epoch 9; iter: 0; batch classifier loss: 0.322431
epoch 10; iter: 0; batch classifier loss: 0.482034


epoch 11; iter: 0; batch classifier loss: 0.590706
epoch 12; iter: 0; batch classifier loss: 0.550292
epoch 13; iter: 0; batch classifier loss: 0.383388
epoch 14; iter: 0; batch classifier loss: 0.468292
epoch 15; iter: 0; batch classifier loss: 0.445077
epoch 16; iter: 0; batch classifier loss: 0.479713


epoch 17; iter: 0; batch classifier loss: 0.430369
epoch 18; iter: 0; batch classifier loss: 0.480602
epoch 19; iter: 0; batch classifier loss: 0.542811
epoch 20; iter: 0; batch classifier loss: 0.460378
epoch 21; iter: 0; batch classifier loss: 0.489297
epoch 22; iter: 0; batch classifier loss: 0.451357


epoch 23; iter: 0; batch classifier loss: 0.592304
epoch 24; iter: 0; batch classifier loss: 0.463698
epoch 25; iter: 0; batch classifier loss: 0.360012
epoch 26; iter: 0; batch classifier loss: 0.476519
epoch 27; iter: 0; batch classifier loss: 0.485747


epoch 28; iter: 0; batch classifier loss: 0.434838
epoch 29; iter: 0; batch classifier loss: 0.524503
epoch 30; iter: 0; batch classifier loss: 0.554338
epoch 31; iter: 0; batch classifier loss: 0.458578
epoch 32; iter: 0; batch classifier loss: 0.548712


... (261 outputs left out)

epoch 36; iter: 0; batch classifier loss: 0.718144; batch adversarial loss: 0.034451
epoch 37; iter: 0; batch classifier loss: 0.863625; batch adversarial loss: 0.090705
epoch 38; iter: 0; batch classifier loss: 0.599641; batch adversarial loss: 0.030415
epoch 39; iter: 0; batch classifier loss: 0.547092; batch adversarial loss: 0.027229


epoch 40; iter: 0; batch classifier loss: 0.665668; batch adversarial loss: 0.088813
epoch 41; iter: 0; batch classifier loss: 0.631431; batch adversarial loss: 0.087061
epoch 42; iter: 0; batch classifier loss: 0.478328; batch adversarial loss: 0.149531
epoch 43; iter: 0; batch classifier loss: 0.455699; batch adversarial loss: 0.022062


epoch 44; iter: 0; batch classifier loss: 0.513731; batch adversarial loss: 0.020745
epoch 45; iter: 0; batch classifier loss: 0.602888; batch adversarial loss: 0.020535
epoch 46; iter: 0; batch classifier loss: 0.524826; batch adversarial loss: 0.020901
epoch 47; iter: 0; batch classifier loss: 0.574935; batch adversarial loss: 0.019269


epoch 48; iter: 0; batch classifier loss: 0.673062; batch adversarial loss: 0.019051
epoch 49; iter: 0; batch classifier loss: 0.547149; batch adversarial loss: 0.017630


In [9]:
# Show what the grid searches logged (one CSV per run_single_sensitive call under ./results)
import glob
frames = [pd.read_csv(f, index_col=0).assign(source=os.path.basename(f)) for f in [f for f in sorted(glob.glob('results/experiment_*.csv'), key=os.path.getmtime) if os.path.getmtime(f) >= T0]]
logged = pd.concat(frames, ignore_index=True)
logged = logged[logged.model != 'model']
logged[['model','BM','balanced_acc','acc','spd','fair_score','source']]

,model,BM,balanced_acc,acc,spd,fair_score,source
0,TabNetClassifier,baseline,0.737286,0.792673,-0.219608,0.935,experiment_(2026_11_05-06_00_31_PM).csv
1,TabNetClassifier,preReweighing posCalibrated,0.737286,0.792673,-0.219608,0.935,experiment_(2026_11_05-06_00_31_PM).csv
2,TabNetClassifier,preReweighing posEqqOds,0.737286,0.792673,-0.219608,0.935,experiment_(2026_11_05-06_00_31_PM).csv
3,TabNetClassifier,preReweighing posROC,0.737286,0.792673,-0.219608,0.935,experiment_(2026_11_05-06_00_31_PM).csv
4,TabNetClassifier,preDisparate posCalibrated,0.737286,0.792673,-0.219608,0.935,experiment_(2026_11_05-06_00_31_PM).csv
...,...,...,...,...,...,...,...
81,<aif360.algorithms.inprocessing.adversarial_de...,preDisparate,0.741225,0.795012,-0.225098,0.950,experiment_(2026_11_05-06_09_42_PM).csv
82,<aif360.algorithms.inprocessing.adversarial_de...,posCalibrated,0.737479,0.790335,-0.229804,0.969,experiment_(2026_11_05-06_09_42_PM).csv
83,<aif360.algorithms.inprocessing.adversarial_de...,posEqqOds,0.738324,0.793453,-0.220392,0.937,experiment_(2026_11_05-06_09_42_PM).csv
84,<aif360.algorithms.inprocessing.adversarial_de...,posROC,0.738324,0.793453,-0.220392,0.937,experiment_(2026_11_05-06_09_42_PM).csv
